<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Penalized ERM and Ridge Regression

Recall our least-squares setup:
$$
\mathcal D=\{(x_n,y_n)\}_{n=1}^N,
\qquad x_n\in\mathbb R^D,
$$
and consider the linear score function
$$
s_w(x)=x^\top w.
$$
Collect the inputs in $X\in\mathbb R^{N\times D}$ and the responses in $y\in\mathbb R^N$. The squared-error empirical risk is
$$
\widehat R(w)=\frac1N\sum_{n=1}^N(y_n-x_n^\top w)^2
=\frac1N\|y-Xw\|_2^2.
$$
Ordinary least squares (OLS) chooses
$$
\widehat w_{\mathrm{OLS}}=\arg\min_w\widehat R(w).
$$
OLS minimizes training error, but the fitted coefficients can be sensitive to the particular training sample, especially with many or highly correlated predictors.

## Regularization and Penalized ERM

**Regularization** is any modification of the learning procedure intended to control flexibility, improve stability, and improve generalization. We have already regularized by restricting a function class and by selecting polynomial degree or nearest-neighbor count $m$.

Penalized ERM changes the fitting objective itself. If $\Omega(s)$ measures model complexity, define
$$
\widehat s_\lambda
=\arg\min_s
\left\{\widehat R(s)+\lambda\Omega(s)\right\},
\qquad \lambda\geq0.
$$

Or if the model is parameterized by $w$, let $\Omega(w)$ measure complexity, define
$$
\widehat w_\lambda
=\arg\min_w
\left\{\widehat R(w)+\lambda\Omega(w)\right\},
\qquad \lambda\geq0.
$$

The tuning parameter $\lambda$ controls the tradeoff:

- $\lambda=0$ recovers ordinary ERM;
- increasing $\lambda$ gives the penalty more influence relative to training fit.


## Ridge Regression

Ridge regression modifies OLS to use a squared "$\ell_2$" penalty to measure complexity
$$
\Omega(w)=\|w\|_2^2=\sum_{j=1}^D w_j^2.
$$
(Here, the subscript in $\|\cdot\|_2$ denotes its the typical Euclidean norm.)
Consequently, the objective is 
$$
\boxed{
J_\lambda(w)=\|y-Xw\|_2^2+\lambda\|w\|_2^2
}
$$
and the best estimator is
$$
\widehat w_\lambda=\arg\min_w J_\lambda(w).
$$
Larger $\lambda$ makes large coefficient vectors more costly and generally moves the fitted coefficients toward zero. With $\lambda = 0$ we get back OLS, while $\lambda \to \infty$ drives $w \to 0$. 

(*Note*: Sometimes people use a slightly different objective, e.g. multiplying the squared-error term by $1/N$. This only rescales the meaning of $\lambda$; scikit-learn calls this `alpha`. Also, the intercept is ordinarily left unpenalized. WLOG assume $y$ has been mean-centered.)

### Why Penalize Coefficient Size?

Coefficient magnitude depends on feature units, which is why ridge is ordinarily applied after standardizing the predictors. Why is $\Omega$ reasonable? 

1. **Large coefficients correspond to more flexible fits.**  
  If the entries of $w$ are large, the model can change rapidly with small changes in $x$. This somehow captures flexibility. Conversely, small coefficients give smoother, more stable models. It limits how strongly any single feature can influence the prediction. 

2. **Handles multicollinearity.**  
  When features are highly correlated, we'll show that OLS can often produce large coefficients that cancel each other out. Penalizing $\|w\|_2^2$ discourages this behavior and leads to more stable solutions.

3. **It's Mathematically convenient.**  
  The squared $\ell_2$ norm is smooth and leads to a closed-form solution, which makes ridge regression easy to analyze and compute.



## Solving the Ridge Problem

Expand the objective:
$$
\begin{aligned}
J_\lambda(w)
&=(y-Xw)^\top(y-Xw)+\lambda w^\top w\\
&=y^\top y-2w^\top X^\top y
  +w^\top X^\top Xw+\lambda w^\top w.
\end{aligned}
$$
Differentiate with respect to $w$:
$$
\nabla_wJ_\lambda(w)
=-2X^\top y+2X^\top Xw+2\lambda w.
$$
Setting the gradient equal to zero gives the **ridge normal equations**
$$
(X^\top X+\lambda I_D)w=X^\top y,
$$
where $I_D$ is the $D\times D$ identity matrix. Thus, for $\lambda>0$,
$$
\boxed{
\widehat w_\lambda
=(X^\top X+\lambda I_D)^{-1}X^\top y.
}
$$
Compare this with the familiar OLS formula: ridge adds $\lambda$ to the diagonal before solving. The result is a different fitting rule, even though the resulting prediction function is still linear in $x$.

For OLS we had to worry about whether the matrix $X^\top X$ was invertible. However, for ridge, if $\lambda > 0$, the matrix $(X^\top X + \lambda I)$ is always invertible. Compare this to the OLS solution:

- OLS requires $X^\top X$ to be invertible (problems when $D>N$ or if features exactly/close-to-exactly colinear/correlated)
- Ridge replaces it with $X^\top X + \lambda I$

Adding $\lambda I$
- ensures invertibility even when $D > N$  
- stabilizes the solution when features are correlated

Often we say this is *better* **conditioned**.

### Correlated Predictors

Suppose two feature columns are nearly identical:
$$
X_{:2}\approx X_{:1}.
$$
Their contribution to the fitted values is
$$
w_1X_{:1}+w_2X_{:2}
\approx(w_1+w_2)X_{:1}.
$$
The pairs $(w_1,w_2)=(1,0)$ and $(100,-99)$ can therefore give similar fitted values. The data constrain $w_1+w_2$ much more strongly than they constrain the two coefficients separately.

Ridge strongly prefers the first pair because
$$
1^2+0^2=1,
\qquad
100^2+(-99)^2=19801.
$$
It discourages large opposing coefficients and produces a more stable allocation of weight among correlated predictors.

## Penalized and Constrained Forms

Ridge can also be written as a constrained problem:
$$
\min_w\|y-Xw\|_2^2
\qquad\text{subject to}\qquad
\|w\|_2^2\leq t.
$$
The parameter $t$ places a hard limit on coefficient size. Smaller $t$ means stronger regularization.

The penalized and constrained formulations trace the same set of ridge solutions: a penalized solution at a given $\lambda$ is also a constrained solution when $t=\|\widehat w_\lambda\|_2^2$. (You may have seen a simplified version of this in a multivariable calculus course under the name **Lagrange multipliers**, more generally this is something known as the primal/dual formulations.)

### Geometry

In two dimensions, $\|w\|_2^2\leq t$ is a disk. The constrained ridge solution is the point in that disk lying on the smallest attainable squared-error contour.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.patches import Circle
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
#| code-fold: true
X_geom = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0],
])
y_geom = np.array([1.0, 1.0, 2.0, 2.5])

def squared_error_geom(w1, w2):
    predictions = X_geom[:, 0, None, None]*w1 + X_geom[:, 1, None, None]*w2
    residuals = y_geom[:, None, None] - predictions
    return np.sum(residuals**2, axis=0)

def ridge_solution_geom(lam):
    return np.linalg.solve(
        X_geom.T @ X_geom + lam*np.eye(2),
        X_geom.T @ y_geom,
    )

w_ols_geom = np.linalg.solve(X_geom.T @ X_geom, X_geom.T @ y_geom)

def constrained_solution_geom(t):
    if w_ols_geom @ w_ols_geom <= t:
        return w_ols_geom

    low, high = 0.0, 1.0
    while ridge_solution_geom(high) @ ridge_solution_geom(high) > t:
        high *= 2
    for _ in range(60):
        middle = (low + high)/2
        w_middle = ridge_solution_geom(middle)
        if w_middle @ w_middle > t:
            low = middle
        else:
            high = middle
    return ridge_solution_geom((low + high)/2)

grid = np.linspace(-2.2, 2.2, 300)
W1, W2 = np.meshgrid(grid, grid)
contours = squared_error_geom(W1, W2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, t in zip(axes, [1.2, 0.35]):
    ax.contour(W1, W2, contours, levels=12, colors="gray", linewidths=0.8)
    ax.add_patch(Circle((0, 0), np.sqrt(t), fill=False,
                        color="tab:blue", linewidth=2))
    w_t = constrained_solution_geom(t)
    ax.scatter(*w_ols_geom, marker="*", s=120, color="black", label="OLS")
    ax.scatter(*w_t, s=55, color="crimson", label="ridge")
    ax.plot([0, w_t[0]], [0, w_t[1]], color="crimson", linewidth=1)
    ax.set(xlabel=r"$w_1$", ylabel=r"$w_2$", title=rf"Constraint: $\|w\|_2^2\leq {t}$",
           xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), aspect="equal")
axes[0].legend()
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
from matplotlib.patches import Circle

In [ ]:
import numpy as np

X = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0]
])

y = np.array([1.0, 1.0, 2.0, 2.5])

w_ols = np.linalg.solve(X.T @ X, X.T @ y)
w_ols

In [ ]:
def risk(w1, w2):
    #vectorized
    pred = X[:, 0, None, None] * w1 + X[:, 1, None, None] * w2
    residuals = y[:, None, None] - pred
    return np.mean(residuals**2, axis=0)  

def penalized_risk(w1, w2, lam):
    #vectorized
    return risk(w1, w2) + lam * (w1**2 + w2**2) 
    
def ridge_solution(lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y)

def constrained_solution(t):
    # approximate mapping correpsondence for lambda given t

    # if it sats constr, lambda=0 (OLS)
    ols_norm_sq = np.dot(w_ols, w_ols)
    if ols_norm_sq <= t:
        return w_ols, 0.0

    # map of lambda to ridge to constr value t
    def norm_sq_at_lambda(lam):
        w = ridge_solution(lam)
        return np.dot(w, w)

    # quick and dirty approx
    lo, hi = 0.0, 1.0
    while norm_sq_at_lambda(hi) > t:
        hi *= 2.0

    #lam somewhere between low and high at this point

    # approx binary search for lam-star sat constr exactly
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if norm_sq_at_lambda(mid) > t:
            lo = mid
        else:
            hi = mid

    lam_star = 0.5 * (lo + hi)
    w_star = ridge_solution(lam_star)
    return w_star, lam_star #<- this is what its trying to get given t what is value of lambda

def plot_geometry(mode="constrained", t=1.0, lam=0.5):
    xlim = (-2.5, 2.5)
    ylim = (-2.5, 2.5)

    fig, ax = plt.subplots(figsize=(7, 7))

    # grid of ws
    w1 = np.linspace(xlim[0], xlim[1], 400)
    w2 = np.linspace(ylim[0], ylim[1], 400)
    W1, W2 = np.meshgrid(w1, w2)

    # OLS risk
    Z = risk(W1, W2)
    levels = np.linspace(np.min(Z) + 0.2, np.min(Z) + 8.0, 12)
    ax.contour(W1, W2, Z, levels=levels, linewidths=1.2)

    if mode == "constrained":
        # add constr circle
        radius = np.sqrt(t)
        circle = Circle((0, 0), radius, fill=False, linestyle='--', linewidth=2)
        ax.add_patch(circle)

        # find lambda corresp to constr value t
        w_con, lam_star = constrained_solution(t)
        ax.plot(w_con[0], w_con[1], 'o', markersize=8, label='Constrained solution')
        ax.plot([0, w_con[0]], [0, w_con[1]], linewidth=1)

        title = (
            r'Constrained form: $\min_w \hat R(w)$ s.t. $\|w\|_2^2 \leq t$'
            + "\n"
            + rf'$t = {t:.2f}$, implied $\lambda^* \approx {lam_star:.3f}$'
        )

    elif mode == "penalized":
        # draw pen risk contours
        Zp = penalized_risk(W1, W2, lam)
        levels_p = np.linspace(np.min(Zp) + 0.05, np.min(Zp) + 5.0, 10)
        ax.contour(W1, W2, Zp, levels=levels_p, linestyles='dashed', linewidths=1.0, colors='red')

        # add ridge soln
        w_ridge = ridge_solution(lam)
        ax.plot(w_ridge[0], w_ridge[1], 'o', markersize=8, label='Penalized solution')

        radius = np.linalg.norm(w_ridge)
        circle = Circle((0, 0), radius, fill=False, linestyle='--', linewidth=2)
        ax.add_patch(circle)

        title = (
            r'Penalized form: $\min_w \hat R(w) + \lambda \|w\|_2^2$'
            + "\n"
            + rf'$\lambda = {lam:.2f}$, implied $t \approx \|w_\lambda\|_2^2 = {radius**2:.3f}$'
        )

    else:
        raise ValueError("mode must be 'constrained' or 'penalized'")

    ax.plot(w_ols[0], w_ols[1], '*', markersize=8, label='OLS solution')
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(r'$w_1$')
    ax.set_ylabel(r'$w_2$')
    ax.set_title(title)
    ax.legend()
    
    plt.show()

In [ ]:
interact(
    plot_geometry,
    mode=Dropdown(
        options=["constrained", "penalized"],
        value="constrained",
        description="view"
    ),
    t=FloatSlider(
        value=1.0, min=0.05, max=6.0, step=0.05,
        description='t'
    ),
    lam=FloatSlider(
        value=0.5, min=0.0, max=50.0, step=0.05,
        description='lambda'
    )
);

## Orthonormal Predictors

The effect of ridge is especially transparent when the columns of $X$ are orthonormal:
$$
X^\top X=I_D.
$$
Then
$$
\begin{aligned}
\widehat w_\lambda
&=(I_D+\lambda I_D)^{-1}X^\top y\\
&=\frac1{1+\lambda}X^\top y\\
&=\frac1{1+\lambda}\widehat w_{\mathrm{OLS}}.
\end{aligned}
$$
Every OLS coefficient is multiplied by the same shrinkage factor. Therefore
$$
\widehat w_\lambda\longrightarrow\widehat w_{\mathrm{OLS}}
\quad\text{as }\lambda\to0,
\qquad
\widehat w_\lambda\longrightarrow0
\quad\text{as }\lambda\to\infty.
$$
This common shrinkage factor is special to the orthonormal case, but it cleanly illustrates what the penalty does.

## The Ridgeless Limit

When least squares has multiple coefficient minimizers, ridge selects a particular limiting solution as $\lambda\to0^+$. If
$$
\mathcal W_{\mathrm{OLS}}
=\arg\min_u\|y-Xu\|_2^2,
$$
then
$$
\boxed{
\widehat w_{0^+}
=\arg\min_{w\in\mathcal W_{\mathrm{OLS}}}\|w\|_2.
}
$$
Thus the ridgeless limit is the minimum-norm least-squares solution. When exact interpolation is possible, the same statement becomes
$$
\widehat w_{0^+}
=\arg\min_w\|w\|_2
\qquad\text{subject to}\qquad Xw=y.
$$
Numerical least-squares routines commonly return this minimum-norm solution when the coefficient vector is not unique.

## Ridge and the Bias–Variance Tradeoff

Increasing $\lambda$ makes the fitted coefficients less responsive to the particular training sample. This often reduces variance while introducing additional bias.

- Small $\lambda$ gives weak shrinkage and can retain high variance.
- Intermediate $\lambda$ can improve performance on new data.
- Very large $\lambda$ drives the slopes toward zero and can underfit.

This often produces a U-shaped evaluation-error curve, although the exact shape in any sample is not a theorem. The following teaching simulation combines many predictors, correlated copies, and response noise. Its held-out curve illustrates the pattern; looking across the test curve here is part of the demonstration, not a valid procedure for selecting $\lambda$.

In [ ]:
rng = np.random.default_rng(90)
N_sim = 140
groups = 20
copies_per_group = 5
D_sim = groups*copies_per_group

Z_sim = rng.normal(size=(N_sim, groups))
X_sim = np.hstack([
    Z_sim[:, [j]] + 0.08*rng.normal(size=(N_sim, copies_per_group))
    for j in range(groups)
])
signal_weights = np.zeros(groups)
signal_weights[:5] = [3.0, -2.0, 1.5, 0.75, -1.0]
y_sim = Z_sim @ signal_weights + rng.normal(scale=2.0, size=N_sim)

X_sim_train, X_sim_test, y_sim_train, y_sim_test = train_test_split(
    X_sim, y_sim, test_size=0.4, random_state=90
)

lambda_sim = np.logspace(-4, 5, 140)
train_mse_sim, test_mse_sim, norm_sim = [], [], []

for lam in lambda_sim:
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=lam, fit_intercept=True, solver="lsqr"),
    )
    model.fit(X_sim_train, y_sim_train)
    train_mse_sim.append(mean_squared_error(y_sim_train, model.predict(X_sim_train)))
    test_mse_sim.append(mean_squared_error(y_sim_test, model.predict(X_sim_test)))
    norm_sim.append(np.linalg.norm(model.named_steps["ridge"].coef_))

best_display = int(np.argmin(test_mse_sim))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lambda_sim, train_mse_sim, label="training MSE")
axes[0].plot(lambda_sim, test_mse_sim, label="held-out MSE")
axes[0].scatter(lambda_sim[best_display], test_mse_sim[best_display],
                color="crimson", zorder=3)
axes[0].set(xscale="log", xlabel=r"$\lambda$", ylabel="MSE",
            title="Fit and held-out performance")
axes[0].legend()

axes[1].plot(lambda_sim, norm_sim)
axes[1].set(xscale="log", xlabel=r"$\lambda$",
            ylabel=r"$\|\widehat w_\lambda\|_2$",
            title="Coefficient shrinkage")
fig.tight_layout()
plt.show()

Training MSE is smallest near the OLS end of the path, while coefficient norm decreases as $\lambda$ grows. In this sample, moderate shrinkage improves held-out performance before excessive shrinkage loses too much signal.

## Choosing $\lambda$ with Cross-Validation

In practice, choose $\lambda$ using validation or cross-validation rather than inspecting the test curve. Because scaling affects the ridge penalty, scaling and fitting belong in the same pipeline and must be repeated within every CV training fold.

We will use a simple workflow:

1. reserve a test set;
2. use CV within the remaining data to select $\lambda$;
3. refit the selected pipeline on all model-building data;
4. evaluate once on the test set.

The CV search and refit together form the model-building procedure.

## Riboflavin Data

The riboflavin dataset has only 71 observations but 4,088 gene-expression predictors. This $D>N$ setting makes unregularized coefficient estimates highly underdetermined and gives ridge a natural role. The response is stored as `y`; the remaining columns are predictors.

In [ ]:
data_candidates = [Path("lectures/riboflavin.csv"), Path("riboflavin.csv")]
data_path = next(path for path in data_candidates if path.exists())
riboflavin = pd.read_csv(data_path)

y_ribo = riboflavin["y"].to_numpy(dtype=float)
X_ribo_frame = riboflavin.drop(columns="y")
feature_names = X_ribo_frame.columns.to_numpy()
X_ribo = X_ribo_frame.to_numpy(dtype=float)

print(f"N = {X_ribo.shape[0]}")
print(f"D = {X_ribo.shape[1]}")

In [ ]:
X_build, X_test, y_build, y_test = train_test_split(
    X_ribo, y_ribo, test_size=0.2, random_state=91
)

lambda_grid = np.logspace(-6, 6, 100)
ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge(fit_intercept=True),
)
folds = KFold(n_splits=5, shuffle=True, random_state=92)

search = GridSearchCV(
    ridge_pipeline,
    param_grid={"ridge__alpha": lambda_grid},
    scoring="neg_mean_squared_error",
    cv=folds,
    refit=True,
)
search.fit(X_build, y_build)

selected_lambda = search.best_params_["ridge__alpha"]
final_model = search.best_estimator_
test_mse = mean_squared_error(y_test, final_model.predict(X_test))

print(f"Selected lambda: {selected_lambda:.4g}")
print(f"Final test MSE: {test_mse:.4f}")

In [ ]:
cv_mean = -search.cv_results_["mean_test_score"]
cv_sd = search.cv_results_["std_test_score"]

plt.figure(figsize=(7, 4))
plt.plot(lambda_grid, cv_mean, label="mean CV MSE")
plt.fill_between(lambda_grid, cv_mean-cv_sd, cv_mean+cv_sd,
                 alpha=0.2, label="± 1 fold SD")
plt.axvline(selected_lambda, linestyle="--", color="black",
            label=rf"selected $\lambda={selected_lambda:.3g}$")
plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel("Cross-validated MSE")
plt.title("Selecting ridge regularization")
plt.legend()
plt.show()

The mean CV curve compares candidate procedures. Its minimizing value selects $\lambda$, so the winning CV score is part of model building rather than a final performance estimate. `GridSearchCV` then refits the complete selected pipeline on all model-building observations. The held-out MSE evaluates that refitted result.

The band shows the standard deviation of the five fold scores. Because the fold scores are dependent, it should not automatically be interpreted as a standard error or confidence interval.

### Coefficient Paths

Ridge usually shrinks coefficients continuously rather than setting them exactly to zero. Plotting several paths shows how the fitted coefficients change with $\lambda$.

In [ ]:
#| code-fold: true
selected_ridge = final_model.named_steps["ridge"]
top_k = 10
rng = np.random.default_rng()
sample_idx = rng.choice(selected_ridge.coef_.shape[0], size=top_k, replace=False)

scaler = final_model.named_steps["standardscaler"]
X_build_scaled = scaler.transform(X_build)
coefficient_path = []
for lam in lambda_grid:
    model = Ridge(alpha=lam, fit_intercept=True, solver="lsqr")
    model.fit(X_build_scaled, y_build)
    coefficient_path.append(model.coef_)
coefficient_path = np.asarray(coefficient_path)

plt.figure(figsize=(8, 5))
for j in sample_idx:
    plt.plot(lambda_grid, coefficient_path[:, j], label=feature_names[j])
plt.axhline(0, color="black", linewidth=0.7)
plt.axvline(selected_lambda, color="black", linestyle="--")
plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel(r"$\widehat w_{\lambda,j}$")
plt.title("Ridge coefficient paths")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

All paths move toward zero as regularization becomes very strong, but they generally do not land exactly at zero for a finite $\lambda$. 
## Review Questions

See: @sec-ridge-questions.